# Imports and Notebook Sett5ings


In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

os.chdir("../")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Data Loading and Inspection


In [3]:
df = pd.read_csv("artifacts/data/raw.csv")
df.shape

(15673, 29)

# Data Split


Due to the nature of the dataset; some guardrails were put in place to have a valid split. Also due to insufficient data points at the intial stage, we use just a train and val split

**Guardrails:**

- Dataset is collapse by locality to adequately represent locality distribution in both the train and val slits.
- A stratified sampling technique was adopted to prevent domination by some localities with high number of listings.
- Finally, a price band analysis was done on the two sets to determine if price distribution was equally represented in the two sets


In [4]:
MIN_LISTINGS = 50

locality_counts = df["locality"].value_counts()
rare_localitys = locality_counts[locality_counts < MIN_LISTINGS].index

df["locality_grouped"] = df["locality"].where(
    ~df["locality"].isin(rare_localitys), other="OTHER"
)

In [5]:
train_df, eval_df = train_test_split(
    df, test_size=0.2, random_state=2025, stratify=df["locality_grouped"]
)

In [6]:
def audit_price_bands(df, name):
    print(f"\n{name} price band distribution")
    print(pd.qcut(df["price"], 4).value_counts(normalize=True).sort_index())


audit_price_bands(train_df, "TRAIN")
audit_price_bands(eval_df, "EVAL")



TRAIN price band distribution
price
(159.999, 2200.0]         0.255623
(2200.0, 4500.0]          0.251236
(4500.0, 12750.0]         0.243260
(12750.0, 278846734.0]    0.249880
Name: proportion, dtype: float64

EVAL price band distribution
price
(199.999, 2400.0]          0.251037
(2400.0, 5000.0]           0.279426
(5000.0, 12598.5]          0.219458
(12598.5, 1655500000.0]    0.250080
Name: proportion, dtype: float64


In [ ]:
train_df.to_csv("artifacts/data/01-raw/train.csv", index=False)
eval_df.to_csv("artifacts/data/01-raw/eval.csv", index=False)

In [1]:
import pandas as pd

data = pd.read_csv("../artifacts/feature_engineering/features_train.csv")